# Simple ResNet Model

## Importing Packages

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchinfo
from torchvision import transforms
from torchvision.datasets import ImageFolder

from torch.utils.data import Dataset, DataLoader, random_split

import helper_utils

# Set seed
SEED = 42

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
dataset_path = "data"

In [26]:
# Display the image count statistics for each class.
helper_utils.display_dataset_stats(dataset_path)

Class Name,Number of Images
Agriculture,800
Airport,800
Beach,800
City,800
Desert,800
Forest,800
Grassland,800
Highway,800
Lake,800
Mountain,800


In [27]:
# Pre-calculated mean and std of this dataset
mean = [0.378, 0.393, 0.345]
std = [0.205, 0.173, 0.170]

# Transforms for Training data with augmentations
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size = (100, 100), scale = (0.8, 1.0)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.RandomVerticalFlip(p = 0.5),
    transforms.RandomRotation(degrees = 15),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.2, hue = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)
])

# Transforms for Validation and Test data (no augmentations)
val_transforms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)
])

In [28]:
skyview_dataset = ImageFolder(root = dataset_path, transform = None)

# Calculate the number of samples for training, validation and testing
total_samples = len(skyview_dataset)

train_size = int (0.7 * total_samples)
val_size = int(0.15 * total_samples)
test_size = total_samples - train_size - val_size

# Split the dataset into training, validation and testing sets
train_data, val_data, test_data = random_split(skyview_dataset, [train_size, val_size, test_size], generator = torch.Generator().manual_seed(SEED))

# Dataset Class

In [29]:
class ImgDataset(Dataset):
    def __init__ (self, data, transform = None):
        super().__init__()

        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        img, label = self.data[index]

        if self.transform:
            img = self.transform(img)

        return img, label

In [30]:
train_dataset = ImgDataset(data = train_data, transform = train_transforms)
val_dataset = ImgDataset(data = val_data, transform = val_transforms)
test_dataset = ImgDataset(data = test_data, transform = val_transforms)

num_classes = 15

## DataLoaders

In [31]:
BATCH_SIZE = 32

train_loader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(dataset = test_dataset, batch_size= 1, shuffle = False)

# Simple ResNet Components

## Residual Block

In [ ]:
# class ResidualBlock(nn.Module):
#     def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
#         # Initialize the parent nn.Module class
#         super(ResidualBlock, self).__init__()

#         # Fist component of the main path
#         self.convblock1 = nn.Sequential(
#             nn.Conv2d(
#                 in_channels = in_channels,
#                 out_channels = out_channels,
#                 kernel_size = 3,
#                 stride = stride,
#                 padding = 1,
#                 bias = False
#             ),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace = True)
#         )

#         # Second component of the main path
#         self.convblock2 = nn.Sequential(
#             nn.Conv2d(
#                 in_channels = out_channels,
#                 out_channels = out_channels,
#                 kernel_size = 3,
#                 stride = 1,
#                 padding = 1,
#                 bias = False
#             ),
#             nn.BatchNorm2d(out_channels),

#         )

#         # Optional downsampling layer for the skip connection
#         self.downsample = downsample

#     def _initial_forward(self, x):
#         out = self.convblock1(x)
#         out = self.convblock2(out)

#         return out

#     def forward(self, x):

#         # stores the input for the skip connection
#         identity = x

#         out = self._initial_forward(x)

#         # Apply downsampling to the identity if needed to match out dimensions
#         if self.downsample is not None:
#             identity = self.downsample(x)

#         out += identity

#         # Apply final ReLU activation after adding the skip connection
#         out = F.relu(out)

#         return out


## Simple ResNet Class

In [ ]:
# class SimpleResNet(nn.Module):
#     def __init__(self, num_classes = 5, num_blocks = [2, 2, 2]):
#         super(SimpleResNet, self).__init__()

#         # Define the number of classes and blocks for each stage of the ResNet architecture
#         # Define the final number of classes for the output layer
#         self.num_classes = num_classes

#         # Define the number of blocks for each stage of the ResNet architecture (e.g., [2, 2, 2] for ResNet-18)
#         self.num_blocks = num_blocks

#         self.in_channels = 32

#         # Define the initial convolutional block
#         self.initial_block = self._get_initial_block()

#         # Define the residual blocks for the ResNet architecture
#         self.res_block1 = self._make_residual_block(out_channels = 32, num_blocks = self.num_blocks[0], stride = 1)
#         self.res_block2 = self._make_residual_block(out_channels = 64, num_blocks = self.num_blocks[1], stride = 2)
#         self.res_block3 = self._make_residual_block(out_channels = 128, num_blocks = self.num_blocks[2], stride = 2)

#         # Define the final classification block
#         self.final_block = self._get_final_block()
        

        


#     def _make_residual_block(self, out_channels, num_blocks, stride):

#         downsample = None

#         if stride != 1 or self.in_channels != out_channels:
#             downsample = nn.Sequential(
#                 nn.Conv2d(self.in_channels, 
#                           out_channels, 
#                           kernel_size = 1,
#                           stride = stride,
#                           bias = False),
#                 nn.BatchNorm2d(out_channels)
#             )

#         layers = []

#         first_block = ResidualBlock(in_channels = self.in_channels, out_channels = out_channels, stride = stride, downsample = downsample)

#         layers.append(first_block)

#         self.in_channels = out_channels

#         for _ in range(1, num_blocks):
#             layers.append(ResidualBlock(in_channels = out_channels, out_channels = out_channels))

#         return nn.Sequential(*layers)
        
#     def _get_initial_block(self):
#         initial_block = nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1, bias = False),
#             nn.BatchNorm2d(32),
#             nn.ReLU(inplace = True)
#         )
#         return initial_block
        
#     def _get_final_block(self):
#         final_block = nn.Sequential(
#             nn.AdaptiveAvgPool2d((1, 1)),
#             nn.Flatten(),
#             nn.Linear(self.in_channels, self.num_classes)
#         )
#         return final_block
    
#     def forward(self, x):
#         x = self.initial_block(x)

#         x = self.res_block1(x)
#         x = self.res_block2(x)
#         x = self.res_block3(x)

#         x = self.final_block(x)

#         return x


## Model Initialization  

In [ ]:
# torch.manual_seed(SEED)

# model = SimpleResNet(num_classes = num_classes)


## Model Architecture Verification

In [ ]:
# # Define a configuration dictionary to store parameters for the model summary.
# config = {
#     "input_size": (BATCH_SIZE, 3, 64, 64),
#     "attr_names": ["input_size", "output_size", "num_params", "trainable"],
#     "col_names_display": ["Input Shape ", "Output Shape", "Param #", "Trainable"],
#     "depth": 3
# }

# # Generate the model summary object using torchinfo with the specified configuration.
# summary = torchinfo.summary(
#     model=model, 
#     input_size=config["input_size"], 
#     col_names=config["attr_names"], 
#     depth=config["depth"]
# )

# # Display the summary as a styled HTML table.
# print("--- Model Summary ---\n")
# helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Model Summary ---



Layer (type:depth-idx),Input Shape,Output Shape,Param #,Trainable
SimpleResNet,"[32, 3, 64, 64]","[32, 15]",--,True
Sequential: 1-1,"[32, 3, 64, 64]","[32, 32, 64, 64]",--,True
Conv2d: 2-1,"[32, 3, 64, 64]","[32, 32, 64, 64]",864,True
BatchNorm2d: 2-2,"[32, 32, 64, 64]","[32, 32, 64, 64]",64,True
ReLU: 2-3,"[32, 32, 64, 64]","[32, 32, 64, 64]",--,--
Sequential: 1-2,"[32, 32, 64, 64]","[32, 32, 64, 64]",--,True
ResidualBlock: 2-4,"[32, 32, 64, 64]","[32, 32, 64, 64]",--,True
Sequential: 3-1,"[32, 32, 64, 64]","[32, 32, 64, 64]","9,280",True
Sequential: 3-2,"[32, 32, 64, 64]","[32, 32, 64, 64]","9,280",True
ResidualBlock: 2-5,"[32, 32, 64, 64]","[32, 32, 64, 64]",--,True


### Second attempt at defining the model and generating the summary with the correct variable names.

In [37]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample = None, stride = 1):
        super(ResidualBlock, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.downsample = downsample


        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size = 1, stride = stride, bias = False),
                nn.BatchNorm2d(out_channels)
            )

        self.convblock1 = nn.Sequential(
            nn.Conv2d(
                in_channels = self.in_channels,
                out_channels = self.out_channels,
                kernel_size = 3,
                stride = self.stride,
                padding = 1,
                bias = False
            ),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(in_place = True)
        )

        self.convblock2 = nn.Sequential(
            nn.Conv2d(
                in_channels = self.out_channels,
                out_channels = self.out_channels,
                kernel_size=3,
                stride = 1,
                padding = 1,
                bias = False
            ),
            nn.BatchNorm2d(self.out_channels)
        )

    def _initial_forward(self, x):

        identity = x

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = self.convblock1(identity)
        out = self.convblock2(out)

        return identity, out
    
    def forward(self, x):
        identity, out = self._initial_forward(x)

        out += identity

        out = F.relu(out)

        return out

In [ ]:
class ResNet(nn.Module):
    def __init__(self, num_classes, in_channels = 32, num_blocks = [2, 2, 2]):

        self.num_classes = num_classes
        self.in_channels = in_channels
        self.num_blocks = num_blocks
        

    def _create_initial_block(self):
        initial_block = nn.Sequential(
            nn.Conv2d(
                in_channels = 3, 
                out_channels = self.in_channels,
                kernel_size = 3,
                stride = 1,
                padding = 1,
                bias = False),

            nn.BatchNorm2d(self.in_channels),
            nn.ReLU(inplace=True)
        )